In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, when, lit, current_timestamp
from datetime import datetime, date
import requests
import pandas as pd
import io

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
boe_base_rate_url = (
    "https://www.bankofengland.co.uk/boeapps/database/_iadb-fromshowcolumns.asp"
    "?csv.x=yes&SeriesCodes=IUMABEDR&CSVF=TN&UsingCodes=Y"
    "&Datefrom=01/Jan/2004&Dateto=01/Jan/2030"
)

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(boe_base_rate_url, headers=headers, timeout=30)
print(f"Status: {response.status_code}")
print(f"Bytes received: {len(response.content):,}")
print(response.text[:500])

In [0]:
pdf = pd.read_csv(io.BytesIO(response.content))

print(f"Rows: {len(pdf)}")
print(pdf.head())
print(pdf.dtypes)

In [0]:
df = spark.createDataFrame(pdf)

df_bronze = df \
    .withColumnRenamed("IUMABEDR", "base_rate_pct") \
    .withColumn("source_url", lit(boe_base_rate_url)) \
    .withColumn("ingested_at", current_timestamp())

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.boe_base_rate")
)

print(f"Written {df_bronze.count():,} rows to bronze.boe_base_rate")

In [0]:
# Sanity check - confirm date ordering and rate values look historically accurate
spark.sql("""
    SELECT DATE, base_rate_pct
    FROM bronze.boe_base_rate
    ORDER BY DATE DESC
    LIMIT 5
""").show()

In [0]:
# Bronze ingest complete
# Source: BoE base rate (series IUMABEDR)
# Rows written: 268 (monthly, Jan 2004 - latest)
# Table: bronze.boe_base_rate
# Note: BoE endpoint requires User-Agent header to bypass Akamai block
# IUMABEDR renamed to base_rate_pct at bronze for readability
# Silver will: cast DATE from string to date type, derive year_month join key

print("bronze_03_boe_base_rate complete.")